# Collaborative Filtering: ALS (Implicit Feedback)

In this notebook we build and evaluate an **Alternating Least Squares (ALS)** collaborative
filtering model on the X-Wines dataset.

Unlike non-personalised recommenders, ALS learns a unique latent-factor vector for every
user and every wine by factorising the user-item confidence matrix.  This means each user
receives a recommendation list that reflects their own rating history.

Metrics, user sampling, and train/test splits are kept **identical** to
`non_personalised_recommenders/popular_recommender.ipynb` so all numbers are directly
comparable across models.


In [1]:
# No extra dependencies — ALS is implemented with numpy and scipy,
# both of which are already listed in requirements.txt.
import sys, importlib
for pkg in ('numpy', 'scipy', 'sklearn'):
    m = importlib.import_module(pkg if pkg != 'sklearn' else 'sklearn')
    print(f'{pkg}: {m.__version__}')

zsh:1: no such file or directory: /Users/oliverholmes/Documents/BCSAI/ThirdYear/Chatbots
zsh:1: no such file or directory: RecEngines/wine-recommendation-system/.venv/bin/python


In [2]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ModuleNotFoundError: No module named 'implicit'

## 1) Environment setup and robust data loading

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for c in [start, *start.parents]:
        if (c / 'dataset').exists() and (c / 'EDA').exists():
            return c
    raise FileNotFoundError('Project root with dataset/ and EDA/ not found')

PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR  = PROJECT_ROOT / 'dataset'
RESULTS_DIR  = PROJECT_ROOT / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

wines = pd.read_csv(
    DATASET_DIR / 'XWines_Full_100K_wines.csv',
    usecols=['WineID', 'WineName', 'Type', 'Country', 'RegionName'],
    dtype={'WineID': 'int32', 'WineName': 'string', 'Type': 'category',
           'Country': 'category', 'RegionName': 'string'}
)

ratings = pd.read_csv(
    DATASET_DIR / 'XWines_Full_21M_ratings.csv',
    usecols=['UserID', 'WineID', 'Rating'],
    dtype={'UserID': 'int32', 'WineID': 'int32', 'Rating': 'float32'}
)

wines   = wines.drop_duplicates(subset='WineID').copy()
ratings = ratings.drop_duplicates(subset=['UserID', 'WineID']).copy()
ratings = ratings[ratings['WineID'].isin(wines['WineID'])].copy()

print('wines:  ', wines.shape)
print('ratings:', ratings.shape)
print('users:  ', ratings['UserID'].nunique())
print('items:  ', ratings['WineID'].nunique())

## 2) EDA recap

Key findings from the EDA notebook that directly shape our CF design choices:

- **Sparsity ≈ 99.98 %** — the full user-item matrix has ~106 B possible cells but only ~21 M
  are filled.  We must store it as a sparse CSR matrix and never materialise a dense version.
- **Ratings are skewed high** (mean 3.885, median 4.0) — we treat ≥ 4.0 as a *positive*
  interaction and encode the raw rating as an ALS *confidence weight* so a 5.0 signals
  stronger preference than a 4.0.
- **Long-tail on both users and wines** — a handful of wines attract the vast majority of
  ratings.  ALS L2 regularisation penalises large factor norms and reduces the tendency to
  only recommend already-popular items.
- **Temporal growth** — rating volume increased year-on-year, but for ranking-quality
  evaluation a random per-user split is sufficient for this notebook.


In [ ]:
item_pop = ratings['WineID'].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ratings['Rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C78A8')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')

item_pop.head(100).reset_index(drop=True).plot(ax=axes[1], color='#F58518')
axes[1].set_title('Top-100 Wine Popularity Curve')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Ratings count')

wines['Country'].value_counts().head(15).plot(kind='bar', ax=axes[2], color='#54A24B')
axes[2].set_title('Top-15 Countries in Catalogue')
axes[2].set_xlabel('Country')

plt.tight_layout()
plt.show()

## 3) Positive interactions and CVTT split

We reuse the **same 5 000-user sample** stored by `popular_recommender.ipynb`
(`data/results/nonpers_sampled_users_5000.csv`) so that all models are evaluated on
identical users and metrics are directly comparable.

- Positive threshold: **≥ 4.0** stars
- Eligibility: users with **5 – 250** positive interactions
- Hold-out: **20 %** of each user's positives → test set; the remainder → trainval
- Cross-validation: trainval is further divided into **3 folds** using round-robin
  assignment to preserve each user's interaction distribution across folds.


In [ ]:
POSITIVE_THRESHOLD        = 4.0
MIN_POSITIVE_INTERACTIONS = 5
MAX_POSITIVE_INTERACTIONS = 250
MAX_USERS = 5000
SAMPLED_USERS_PATH = RESULTS_DIR / 'nonpers_sampled_users_5000.csv'

positive    = ratings[ratings['Rating'] >= POSITIVE_THRESHOLD].copy()
user_counts = positive['UserID'].value_counts()

eligible_users = user_counts[
    (user_counts >= MIN_POSITIVE_INTERACTIONS) &
    (user_counts <= MAX_POSITIVE_INTERACTIONS)
].index
positive = positive[positive['UserID'].isin(eligible_users)].copy()

if SAMPLED_USERS_PATH.exists():
    sampled_users = pd.read_csv(SAMPLED_USERS_PATH)['UserID'].to_numpy()
    sampled_users = np.intersect1d(sampled_users, positive['UserID'].unique())
else:
    rng = np.random.default_rng(RANDOM_STATE)
    sampled_users = rng.choice(
        positive['UserID'].unique(),
        size=min(MAX_USERS, positive['UserID'].nunique()),
        replace=False
    )
    pd.Series(sampled_users, name='UserID').to_csv(SAMPLED_USERS_PATH, index=False)

positive = positive[positive['UserID'].isin(sampled_users)].copy()


def split_train_test_per_user(df, test_fraction=0.2, random_state=42):
    tr, te = [], []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state)
        n_test = max(1, int(np.ceil(len(g) * test_fraction)))
        te_g, tr_g = g.iloc[:n_test], g.iloc[n_test:]
        if len(tr_g) == 0:
            continue
        tr.append(tr_g)
        te.append(te_g)
    return pd.concat(tr, ignore_index=True), pd.concat(te, ignore_index=True)


def assign_fold_ids_per_user(df, n_folds=3, random_state=42):
    chunks = []
    for _, g in df.groupby('UserID'):
        g = g.sample(frac=1.0, random_state=random_state).copy()
        g['fold_id'] = np.arange(len(g)) % n_folds
        chunks.append(g)
    return pd.concat(chunks, ignore_index=True)


trainval_pos, test_pos = split_train_test_per_user(
    positive, test_fraction=0.2, random_state=RANDOM_STATE
)
trainval_folds = assign_fold_ids_per_user(trainval_pos, n_folds=3, random_state=RANDOM_STATE)

print('trainval:', trainval_pos.shape)
print('test:    ', test_pos.shape)

## 4) Metrics

Identical to the popularity-baseline notebook so results are comparable across models:

| Metric | Description |
|---|---|
| **Accuracy@K** | Fraction of top-K recommendations that are relevant (hit ratio at list level) |
| **NDCG@K** | Ranking quality with position discount |
| **Diversity@K** | Intra-list diversity via type/country Jaccard distance |
| **Personalisation@10** | Dissimilarity between users' recommendation lists |
| **Coverage** | Share of catalogue that appears in top-10 recommendations across all users |


In [ ]:
def accuracy_at_k(relevant, recommended, k):
    rec_k = recommended[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for x in rec_k if x in relevant)
    return hits / k


def dcg_at_k(relevant, recommended, k):
    s = 0.0
    for i, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            s += 1.0 / np.log2(i + 1)
    return s


def ndcg_at_k(relevant, recommended, k):
    ideal_hits = min(len(relevant), k)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg_at_k(relevant, recommended, k) / idcg

In [ ]:
item_meta = wines.set_index('WineID')[['Type', 'Country']].copy()
item_meta['Type']    = item_meta['Type'].astype(str).str.lower().fillna('unknown')
item_meta['Country'] = item_meta['Country'].astype(str).str.lower().fillna('unknown')

item_signatures = {
    wid: {f"type:{item_meta.loc[wid, 'Type']}", f"country:{item_meta.loc[wid, 'Country']}"}
    for wid in item_meta.index
}


def jaccard_distance(a, b):
    inter = len(a & b)
    union = len(a | b)
    return 0.0 if union == 0 else 1.0 - inter / union


def intra_list_diversity(recommended):
    if len(recommended) < 2:
        return 0.0
    vals = [
        jaccard_distance(
            item_signatures.get(recommended[i], {'unknown'}),
            item_signatures.get(recommended[j], {'unknown'})
        )
        for i in range(len(recommended))
        for j in range(i + 1, len(recommended))
    ]
    return float(np.mean(vals)) if vals else 0.0


def personalisation_at_k(recs_by_user, k=10, max_users=300, random_state=42):
    users = list(recs_by_user.keys())
    if len(users) < 2:
        return 0.0
    rng = np.random.default_rng(random_state)
    if len(users) > max_users:
        users = rng.choice(users, size=max_users, replace=False).tolist()
    sets = {u: set(recs_by_user[u][:k]) for u in users}
    sims = []
    for u1, u2 in itertools.combinations(users, 2):
        a, b = sets[u1], sets[u2]
        union = len(a | b)
        sims.append((len(a & b) / union) if union > 0 else 0.0)
    return float(1.0 - np.mean(sims)) if sims else 0.0

In [ ]:
def evaluate_model(artifact, recommend_fn, eval_users, relevant_dict, ks=(5, 10, 20)):
    rows = []
    recs_by_user   = {}
    max_k          = max(ks)
    all_top10_items = []

    for user_id in eval_users:
        relevant = relevant_dict.get(user_id, set())
        if not relevant:
            continue
        recs = recommend_fn(artifact, user_id, top_k=max_k)
        if not recs:
            continue
        recs_by_user[user_id] = recs
        all_top10_items.extend(recs[:10])
        row = {'UserID': user_id}
        for k in ks:
            row[f'Accuracy@{k}']  = accuracy_at_k(relevant, recs, k)
            row[f'NDCG@{k}']      = ndcg_at_k(relevant, recs, k)
            row[f'Diversity@{k}'] = intra_list_diversity(recs[:k])
        rows.append(row)

    eval_df = pd.DataFrame(rows)
    if eval_df.empty:
        return eval_df, {}, recs_by_user

    coverage = (
        len(set(all_top10_items)) / len(artifact['all_items'])
        if len(artifact['all_items']) else 0.0
    )
    pers = personalisation_at_k(recs_by_user, k=10, max_users=300,
                                 random_state=RANDOM_STATE)
    summary_extra = {'Coverage': coverage, 'Personalisation@10': pers}
    return eval_df, summary_extra, recs_by_user

In [ ]:
def plot_model_diagnostics(eval_df, recs_by_user, model_name, top_k=10):
    if eval_df.empty:
        print(f'No evaluation rows for {model_name}')
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    sns.histplot(eval_df[f'Accuracy@{top_k}'], bins=25, kde=True,
                 ax=axes[0], color='#4C78A8')
    axes[0].set_title(f'{model_name} – Accuracy@{top_k}')

    sns.histplot(eval_df[f'NDCG@{top_k}'], bins=25, kde=True,
                 ax=axes[1], color='#54A24B')
    axes[1].set_title(f'{model_name} – NDCG@{top_k}')

    all_items = [item for recs in recs_by_user.values() for item in recs[:top_k]]
    freq = pd.Series(all_items).value_counts() if all_items else pd.Series(dtype='int64')
    if freq.empty:
        axes[2].text(0.5, 0.5, 'No recs', ha='center', va='center')
    else:
        sns.histplot(freq.values, bins=30, kde=False, ax=axes[2], color='#F58518')
    axes[2].set_title(f'{model_name} – Coverage histogram')
    axes[2].set_xlabel('recommendation count per item')

    plt.tight_layout()
    plt.show()

## 5) ALS model (Implicit Collaborative Filtering)

### How ALS works

Every observed (user, wine) positive rating is converted to a *confidence* value:

$$c_{ui} = 1 + \alpha \cdot r_{ui}$$

ALS decomposes the user-item matrix into low-rank embeddings **U** (*users × factors*)
and **V** (*items × factors*) by minimising the confidence-weighted squared error:

$$\min_{U,V} \sum_{u,i} c_{ui}\,(p_{ui} - u_u^\top v_i)^2 + \lambda(\|U\|^2 + \|V\|^2)$$

where $p_{ui} = 1$ if the user rated the wine and $0$ otherwise.  Crucially, **every**
user-item pair contributes — even unobserved ones (with low confidence $c = 1$) — so the
model learns from both positive evidence and the absence of a rating.

Fixing **V** gives a closed-form solution for each row of **U** (and vice versa):

$$u_u = \left(V^\top V + \sum_{i} (c_{ui}-1)\,v_i v_i^\top + \lambda I\right)^{-1} \sum_i c_{ui}\,v_i$$

Because $c_{ui} - 1 = 0$ for items the user has *not* rated, only the
**rated items** need to be summed — making each update $O(k^2 f)$ where $k$ is
the number of rated items and $f$ the number of factors.
We implement this directly with `numpy.linalg.solve` over scipy CSR matrices —
no compiled C extensions required.

### Parameter choices

| Parameter | Value | Rationale |
|---|---|---|
| `factors` | 64 | Good capacity for 5 K users; reduce if training is slow |
| `regularization` | 0.05 | Slightly stronger than default to handle small user set |
| `iterations` | 15 | Convergence is typical by 10–20 iterations |
| `alpha` | 40 | Confidence scaling; higher → stronger signal from high ratings |


In [ ]:
ALS_FACTORS        = 64
ALS_REGULARIZATION = 0.05
ALS_ITERATIONS     = 15
ALS_ALPHA          = 40   # confidence scaling: c(u,i) = 1 + alpha * rating


def fit_als_model(train_df, all_wine_ids,
                  factors=ALS_FACTORS,
                  regularization=ALS_REGULARIZATION,
                  iterations=ALS_ITERATIONS,
                  alpha=ALS_ALPHA,
                  random_state=RANDOM_STATE):
    """Train confidence-weighted ALS on positive interactions.

    Implemented in pure numpy/scipy — no compiled extensions required.
    Only wines that appear in train_df are given latent factors; wines
    with no training signal are excluded from recommendations.

    Returns an artifact dict compatible with evaluate_model / recommend_als.
    """
    rng = np.random.default_rng(random_state)

    # Re-index to contiguous integers (required by scipy.sparse)
    users        = sorted(train_df['UserID'].unique())
    active_wines = sorted(train_df['WineID'].unique())   # only trained wines
    user2idx = {u: i for i, u in enumerate(users)}
    item2idx = {v: i for i, v in enumerate(active_wines)}

    u_idx    = train_df['UserID'].map(user2idx).values
    i_idx    = train_df['WineID'].map(item2idx).values
    ratings  = train_df['Rating'].values.astype('float32')

    n_users, n_items = len(users), len(active_wines)

    # Sparse user-item matrix (values are raw ratings; confidence applied during ALS)
    user_item = sp.csr_matrix((ratings, (u_idx, i_idx)), shape=(n_users, n_items))
    item_user = user_item.T.tocsr()

    # Initialise item factors with small noise; user factors start at zero
    item_factors = (rng.standard_normal((n_items, factors)) * 0.01).astype('float32')
    user_factors = np.zeros((n_users, factors), dtype='float32')
    reg_eye      = (regularization * np.eye(factors)).astype('float32')

    for it in range(iterations):
        # ── User step ────────────────────────────────────────────────────────
        # Precompute V^T V once; the per-user correction only touches rated items
        YtY = item_factors.T @ item_factors  # (f x f)
        for u in range(n_users):
            s, e = user_item.indptr[u], user_item.indptr[u + 1]
            if s == e:
                continue
            ii   = user_item.indices[s:e]
            conf = 1.0 + alpha * user_item.data[s:e]
            Yi   = item_factors[ii]                         # (k x f)
            # A = V^T V + V_u^T diag(c-1) V_u + λI
            A = YtY + (Yi * (conf - 1.0)[:, None]).T @ Yi + reg_eye
            b = Yi.T @ conf
            user_factors[u] = np.linalg.solve(A, b)

        # ── Item step ────────────────────────────────────────────────────────
        XtX = user_factors.T @ user_factors  # (f x f)
        for i in range(n_items):
            s, e = item_user.indptr[i], item_user.indptr[i + 1]
            if s == e:
                continue
            uu   = item_user.indices[s:e]
            conf = 1.0 + alpha * item_user.data[s:e]
            Xu   = user_factors[uu]                         # (k x f)
            A = XtX + (Xu * (conf - 1.0)[:, None]).T @ Xu + reg_eye
            b = Xu.T @ conf
            item_factors[i] = np.linalg.solve(A, b)

        print(f'  Iteration {it + 1:2d}/{iterations}')

    return {
        'user_factors': user_factors,
        'item_factors': item_factors,
        'user2idx':     user2idx,
        'item2idx':     item2idx,
        'idx2item':     {i: v for v, i in item2idx.items()},
        'user_item':    user_item,
        'all_items':    np.array(sorted(all_wine_ids)),   # full catalogue for coverage
        'train_seen':   train_df.groupby('UserID')['WineID'].apply(set).to_dict(),
    }


def recommend_als(artifact, user_id, top_k=10):
    """Return top-K WineIDs by dot-product score, excluding training-seen wines."""
    user_idx = artifact['user2idx'].get(user_id)
    if user_idx is None:
        return []

    user_vec = artifact['user_factors'][user_idx]           # (f,)
    scores   = (artifact['item_factors'] @ user_vec).copy() # (n_active_items,)

    # Zero-out training items so they are never recommended
    seen = artifact['train_seen'].get(user_id, set())
    for wid in seen:
        ii = artifact['item2idx'].get(wid)
        if ii is not None:
            scores[ii] = -np.inf

    n_valid = int(np.sum(np.isfinite(scores)))
    if n_valid == 0:
        return []
    k = min(top_k, n_valid)
    top_ii = np.argpartition(scores, -k)[-k:]
    top_ii = top_ii[np.argsort(scores[top_ii])[::-1]]
    return [int(artifact['idx2item'][i]) for i in top_ii[:top_k]]

## 6) Cross-validation

Three-fold CV on `trainval_pos` — for each fold we train ALS on two folds and
evaluate on the third, then average metrics across folds to estimate generalisation
performance before touching the held-out test set.


In [ ]:
N_FOLDS      = 3
KS           = (5, 10, 20)
all_wine_ids = wines['WineID'].unique().tolist()

cv_rows = []

for fold in range(N_FOLDS):
    fold_train = trainval_folds[trainval_folds['fold_id'] != fold].copy()
    fold_val   = trainval_folds[trainval_folds['fold_id'] == fold].copy()

    relevant_val = fold_val.groupby('UserID')['WineID'].apply(set).to_dict()
    eval_users   = list(relevant_val.keys())

    artifact = fit_als_model(fold_train, all_wine_ids)
    eval_df, summary_extra, _ = evaluate_model(
        artifact, recommend_als, eval_users, relevant_val, ks=KS
    )

    if eval_df.empty:
        print(f'Fold {fold}: no evaluation rows — skipping')
        continue

    fold_summary = eval_df[[c for c in eval_df.columns if c != 'UserID']].mean().to_dict()
    fold_summary.update(summary_extra)
    fold_summary['fold'] = fold
    cv_rows.append(fold_summary)

    print(
        f'Fold {fold}:  '
        f'Accuracy@10={fold_summary.get("Accuracy@10", 0):.4f}  '
        f'NDCG@10={fold_summary.get("NDCG@10", 0):.4f}  '
        f'Coverage={fold_summary.get("Coverage", 0):.4f}  '
        f'Personalisation@10={fold_summary.get("Personalisation@10", 0):.4f}'
    )

cv_df = pd.DataFrame(cv_rows)
print('\nCV mean across folds:')
print(cv_df[[c for c in cv_df.columns if c != 'fold']].mean().round(4).to_frame('mean'))

In [ ]:
if not cv_df.empty:
    mean_metrics = cv_df[[c for c in cv_df.columns if c != 'fold']].mean()

    groups = {prefix: [] for prefix in ('Accuracy', 'NDCG', 'Diversity')}
    for col in mean_metrics.index:
        for prefix in groups:
            if col.startswith(prefix):
                groups[prefix].append(col)

    colors = ['#4C78A8', '#54A24B', '#F58518']
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    for ax, (label, cols), color in zip(axes, groups.items(), colors):
        vals   = [mean_metrics[c] for c in cols]
        labels = [c.split('@')[1] for c in cols]
        ax.bar(labels, vals, color=color)
        ax.set_title(f'CV mean {label}@K')
        ax.set_xlabel('K')
        ax.set_ylim(0, max(vals) * 1.4 if vals else 1)
        for i, v in enumerate(vals):
            ax.text(i, v + max(vals) * 0.03, f'{v:.4f}', ha='center', fontsize=9)

    plt.suptitle('ALS – 3-fold cross-validation averages', y=1.02)
    plt.tight_layout()
    plt.show()

## 7) Final test evaluation

We now train ALS on the **full `trainval_pos`** set (all three CV folds combined) and
evaluate against the held-out `test_pos` items.  This is the number to report as the
model's final performance and the one to compare against the popularity baselines.


In [ ]:
final_artifact = fit_als_model(trainval_pos, all_wine_ids)

relevant_test   = test_pos.groupby('UserID')['WineID'].apply(set).to_dict()
test_eval_users = list(relevant_test.keys())

test_eval_df, test_summary_extra, test_recs_by_user = evaluate_model(
    final_artifact, recommend_als, test_eval_users, relevant_test, ks=KS
)

print('Test set — per-metric averages:')
print(test_eval_df[[c for c in test_eval_df.columns if c != 'UserID']]
      .mean().round(4).to_frame('mean'))
print('\nTest set — Coverage & Personalisation:')
for k, v in test_summary_extra.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
plot_model_diagnostics(test_eval_df, test_recs_by_user, 'ALS (test set)', top_k=10)

## 8) Summary table

Consolidate all test-set metrics into a single display-friendly table.


In [ ]:
if not test_eval_df.empty:
    metric_cols = [c for c in test_eval_df.columns if c != 'UserID']
    summary = test_eval_df[metric_cols].agg(['mean', 'median', 'std']).round(4)
    summary.loc['Coverage']           = test_summary_extra.get('Coverage', float('nan'))
    summary.loc['Personalisation@10'] = test_summary_extra.get('Personalisation@10', float('nan'))
    display(summary)

## 9) Conclusions

### ALS vs. popularity baseline

Unlike non-personalised recommenders that give every user the same list, ALS builds a
unique latent-factor vector per user, so **Personalisation@10 should be substantially
higher**.  The trade-off is that ALS can still over-recommend popular items because
they appear in many users' training data — this shows up as lower Coverage relative
to what a maximally diverse model would achieve.

### Effect of dataset scale

Training on only 5 000 users limits the collaborative signal — with the full ~1 M
eligible users the latent factors would capture far richer preference patterns and
NDCG/Accuracy are expected to improve.  The 5 K-user setup is intentional here for
fair cross-model comparison within the same hardware constraints.

### Cold-start limitation

Users absent from the training matrix receive no recommendations from `recommend_als`
(the function returns an empty list).  A hybrid approach such as **LightFM** — which
accepts item-side features (Type, Country, Body, Grapes) alongside interaction data —
would address cold-start by falling back to content signals for unseen users.

### Implementation note

ALS is implemented here with `numpy.linalg.solve` over scipy CSR matrices —
no compiled extensions required.  The per-user solve is `O(k² f)` where `k` is
the number of rated items and `f` the number of factors; with 5 K users and
~15 items on average this is fast in pure Python + BLAS.  For larger scale
the `implicit` library (Cython-compiled) provides the same algorithm ~10–50×
faster once its wheel is available for your Python / numpy version.

### Next steps

- **Hyperparameter search**: grid-search `factors ∈ {32, 64, 128}`,
  `regularization ∈ {0.01, 0.05, 0.1}`, `alpha ∈ {10, 20, 40}` over CV folds
- **Scale training** to all eligible users (~1 M) for richer embeddings
- **Compare** against content-based TF-IDF / BERT models on the same metrics
- **LightFM hybrid** to use wine metadata (Type, Country, Body, Grapes) and resolve cold-start
